# Synthetic call-transcript generator — Azure `o4-mini`

Reproduces how the synthetic training corpus was authored: for a chosen
**scenario** and **difficulty tier**, Azure `o4-mini` drafts a realistic, **long**
SP Group call-centre transcript together with its gold PII labels; a **gold-
integrity checker** then verifies every gold value occurs in the text the listed
number of times (the *gold invariant*), and files are saved as
`NNN_tier_scenario.json` in a `data/generated/` staging folder.

This is the runnable version of the hand-authored loop in `DATA.md` (draft →
check → fix → save). The `.py` sibling is a static 3-record shape demo; this
notebook is the live generator.

### Prerequisites
- `pip install -r requirements.txt` (installs `langchain-openai`, used below).
- The standard company `AZURE_OPENAI_*` environment variables (endpoint / API key).

### Workflow
Generation is a **staging** step, not a direct write into the training set:

1. Run the cells below → transcripts land in `data/generated/`.
2. Review them — fix or delete any tagged `_REVIEW` (the checker was unsure).
3. Move the good ones into `data/train/synthetic/` (or `data/val/synthetic/` to
   hold some out), then run `build_splits.py`.

**Data safety:** ships with no saved outputs and only synthetic examples.
`data/generated/` is gitignored, so nothing here is ever committed.

## 0. Config

In [ ]:
from pathlib import Path

# Staging folder for freshly generated transcripts. Review here first, then move
# approved files into data/train/synthetic/ (or data/val/synthetic/).
REPO = Path.cwd()
while REPO.name and not (REPO / 'inference' / 'labels.py').exists():
    REPO = REPO.parent            # find the pii/ root whether run from repo root or here
OUT_DIR = REPO / 'data' / 'generated'

# Filenames continue the corpus numbering: NNN_tier_scenario.json. Bump this so a
# new batch does not overwrite files from an earlier one.
START_INDEX = 1000

# The 9 labels the model is trained on (see inference/labels.py).
LABELS = ['sg_phone_number', 'sg_nric_fin', 'sg_address', 'sg_postal_code',
          'sg_address_unit_number', 'sg_address_block_number', 'email_address',
          'account_number', 'full_name']
OUT_DIR

## 1. Azure `o4-mini` client

Same client as the leak-judge notebook — the company Azure deployment, driven by
the standard `AZURE_OPENAI_*` environment variables (endpoint / API key).

In [ ]:
from langchain_openai import AzureChatOpenAI

def get_o4_mini():
    """Azure o4-mini. Expects the standard AZURE_OPENAI_* environment variables."""
    return AzureChatOpenAI(
        azure_deployment='o4-mini',
        api_version='2024-12-01-preview',
        model_name='o4-mini',
        max_completion_tokens=16000,
        timeout=300,
    )

## 2. Difficulty tiers + the generation prompt

The tier controls **how the PII is spoken** — this variety is the whole point of
the synthetic corpus (it teaches the model that PII arrives whole, in fragments,
digit-by-digit, spelled out, and read back for confirmation).

| tier | what it exercises |
|---|---|
| `normal` | PII stated plainly; maybe one agent read-back |
| `medium` | some spoken-word numbers, light fragmentation, a read-back or two |
| `hard` | heavy fragmentation, digit-by-digit, values split across turns, multiple people, several read-backs |
| `negative` | a genuine call with **no PII at all** (FAQ / general enquiry) — teaches the model not to over-redact |

**Length matters.** Real calls are long and meandering, and the model needs to
cope with that, so the prompt asks for a substantial transcript (the authored
corpus sits around 3,000–4,000 characters). The prompt also enforces the **gold
invariant**: a value spoken/read-back *N* times is listed *N* times.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers.json import JsonOutputParser

TIER_GUIDE = {
    'normal':   'PII is stated plainly and clearly. At most one agent read-back. Natural, unhurried call.',
    'medium':   'Some numbers are spoken as words (\'nine one two three...\') or in small groups; the agent reads back one or two values for confirmation, so those appear twice.',
    'hard':     'PII is fragmented and messy: digits spoken one-by-one, values split across several turns, corrections/restarts, an email spelled letter-by-letter, and MULTIPLE agent read-backs. Optionally two different people (e.g. account holder + joint holder) each with their own NRIC/phone.',
    'negative': 'A realistic call that contains NO personal data at all (general billing FAQ, appliance energy question, opening-hours query). The entities dict MUST be empty.',
}

GEN_PROMPT = """You are generating ONE realistic Singapore SP Group (electricity/utilities)
call-centre transcript for training a PII-redaction model.

Scenario: {scenario}
Difficulty tier: {tier} -- {tier_guide}

Format rules:
- Speaker-labelled turns, alternating: SPEAKER_00 is the agent, SPEAKER_01 is the customer.
- Natural spoken Singlish-tinged English is fine.
- Make it LONG and realistic: roughly 3200-4200 characters, about 18-28 turns, with
  natural small talk, hold/verification moments, and clarifications. Do NOT produce a
  short, clipped call -- length and messiness are the point.
- Use realistic but FAKE Singapore data: 8-digit mobiles starting 8 or 9; NRIC = letter+7 digits+letter (e.g. S1234567A); 6-digit postal codes; HDB block + street + unit (#NN-NNN); emails; 10-digit-ish account numbers.
- Do NOT use any real person's data.

PII label set (use ONLY these keys, omit a key if unused):
  sg_phone_number, sg_nric_fin, sg_address, sg_postal_code,
  sg_address_unit_number, sg_address_block_number, email_address,
  account_number, full_name

GOLD INVARIANT (critical): in 'entities', list each value once per occurrence in the
transcript. If a phone number is spoken by the customer AND read back by the agent, it
appears TWICE in the transcript, so list it twice (once per surface form -- the spoken
words and the digit read-back are different strings, list both). Every listed value must
appear VERBATIM in the transcript text.

Return ONLY this JSON, no prose:
{{"transcript": "<the full speaker-labelled transcript, with \\n between turns>",
  "entities": {{"<label>": ["<exact surface string>", ...]}}}}
"""

gen_chain = PromptTemplate.from_template(GEN_PROMPT) | get_o4_mini() | JsonOutputParser()

## 3. Gold-integrity checker

The step that repeatedly *caught under-counts* during the real authoring. For every
gold value it compares the listed count against the actual occurrence count in the
transcript, and flags very short values (like a bare block number `45`) that may be
substring collisions and need a human eye.

In [ ]:
from collections import Counter

def check_gold_invariant(text, entities):
    """Return a list of human-readable problems; empty list == clean."""
    problems = []
    unknown = set(entities) - set(LABELS)
    if unknown:
        problems.append(f'unknown label key(s): {sorted(unknown)}')
    listed = Counter()
    for label, vals in entities.items():
        for v in vals:
            listed[v] += 1
    for v, k in listed.items():
        occ = text.count(v)
        if occ != k:
            problems.append(f'{v!r}: listed {k}x but occurs {occ}x in transcript')
        if len(v.strip()) <= 2:
            problems.append(f'{v!r}: very short value -- verify it is not a substring collision')
    return problems

## 4. Generate one transcript (with automatic repair)

Draft → check → if the checker complains, show the model its own output plus the
problems and ask it to fix the counts → re-check. Files that still fail after the
repair attempt are saved with a `_REVIEW` marker so a human can finish them, exactly
as in the manual process.

In [ ]:
import json, re

def _slug(scenario):
    return re.sub(r'[^a-z0-9]+', '_', scenario.lower()).strip('_')[:48]

REPAIR_PROMPT = PromptTemplate.from_template(
    'Here is a call transcript and its PII gold labels. A checker found these problems:\n'
    '{problems}\n\nFix ONLY the entities dict so every value is listed exactly as many '
    'times as it occurs in the transcript (the gold invariant). Do not change the transcript.\n\n'
    'transcript:\n{transcript}\n\ncurrent entities:\n{entities}\n\n'
    'Return ONLY the corrected JSON: {{"transcript": <unchanged>, "entities": {{...}}}}')
repair_chain = REPAIR_PROMPT | get_o4_mini() | JsonOutputParser()

def generate_one(scenario, tier, index, save=True):
    out = gen_chain.invoke({'scenario': scenario, 'tier': tier, 'tier_guide': TIER_GUIDE[tier]})
    text, ents = out['transcript'], out.get('entities', {})
    problems = check_gold_invariant(text, ents)
    if problems:
        fixed = repair_chain.invoke({'problems': '\n'.join(problems),
                                     'transcript': text,
                                     'entities': json.dumps(ents, ensure_ascii=False)})
        ents = fixed.get('entities', ents)
        problems = check_gold_invariant(text, ents)
    marker = '' if not problems else '_REVIEW'
    fname = f'{index:03d}_{tier}_{_slug(scenario)}{marker}.json'
    record = {'input': text, 'output': {'entities': ents}}
    if save:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        (OUT_DIR / fname).write_text(json.dumps(record, ensure_ascii=False), encoding='utf-8')
    status = 'OK' if not problems else 'NEEDS REVIEW: ' + '; '.join(problems)
    print(f'{fname:60} {len(text):5d} chars  {sum(len(v) for v in ents.values()):3d} gold  [{status}]')
    return fname, record, problems

## 5. Batch generation

Edit `SCENARIOS` (a `(scenario, tier)` list) to whatever mix you want and run. A
healthy batch mixes tiers and includes some `negative` calls so the model learns not
to over-redact. Output goes to `data/generated/` for review — it is **not** picked up
by training until you move the approved files into `data/train/synthetic/`.

In [ ]:
SCENARIOS = [
    ('customer updates mailing address after moving', 'normal'),
    ('customer updates contact number, agent reads it back', 'medium'),
    ('NRIC update after a legal name change, joint account holder also mentioned', 'hard'),
    ('online account registration, email spelled out letter by letter', 'hard'),
    ('general question about ceiling fan vs aircon electricity usage', 'negative'),
]

results = []
for i, (scenario, tier) in enumerate(SCENARIOS):
    results.append(generate_one(scenario, tier, START_INDEX + i))

review = [f for f, _r, p in results if p]
print(f'\n{len(results)} generated in {OUT_DIR}, {len(review)} need human review: {review}')

## 6. Verify the staging folder

Re-scan `data/generated/` and run the integrity check across everything in it, so you
can confirm what is clean before moving files into the training split.

In [ ]:
clean = flagged = 0
for p in sorted(OUT_DIR.glob('*.json')):
    rec = json.loads(p.read_text(encoding='utf-8'))
    probs = check_gold_invariant(rec['input'], rec['output']['entities'])
    if probs:
        flagged += 1
        print(f'FLAGGED {p.name}: {probs}')
    else:
        clean += 1
print(f'\n{clean} clean, {flagged} flagged in {OUT_DIR}')
print('Next: review/fix flagged files, move approved ones into data/train/synthetic/')
print('      (or data/val/synthetic/), then run finetuning/data_prep/build_splits.py')